# NB03 — Hessen/Darmstadt Inference

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `harish77718/darmstadt-dop20-presliced` — Hessen DOP20 pre-sliced 512×512 PNG patches

**What this does:**
1. Env setup + clone `HarishDeepak/SegEarth-OV-3`
2. Run inference on ~289 non-overlapping patches (17×17 stitch grid)
   - `cls_hessen.txt` — **7 urban classes** matching teammate vocabulary
   - `slide_crop=512` == patch size → single forward pass per patch (no sliding window overhead)
   - `bg_idx=255`, `confidence_threshold=0.15` matching teammate's working config
3. Save `.npy` predictions per patch (stitch cell runs independently — no re-inference on failure)
4. Stitch into `hessen_stitched.png` for visual comparison

> Visual comparison only — no OSM F1 metric for now.

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned → {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Run Hessen inference

Config: `cls_hessen.txt` — **7 urban classes** (impervious surface, building, tree canopy, vehicle, vegetation, railway, road).
Visual only, no GT. `slide_crop=512` == patch size → always `_inference_single_view`, single GPU pass per patch. T4 Tensor Cores active via `bfloat16` autocast.

**Modes (set `STITCH_REGION / N_SAMPLE / SELECTED_PATCHES` below):**
- `STITCH_REGION = True` *(default)*: picks non-overlapping patches from y0–y4352 / x0–x4352 (~289 patches, ~15-25 min on T4), stitches into `hessen_stitched.png`
- `N_SAMPLE = 50`: quick 50-patch sanity check (~5 min)
- `N_SAMPLE = None`: all patches (~2h)
- `SELECTED_PATCHES = [...]`: run specific patches by filename stem

In [ ]:
%%bash
export MPLBACKEND=Agg
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os, time, re
os.environ["MPLBACKEND"] = "Agg"
torch.manual_seed(42); np.random.seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── USER CONFIG ────────────────────────────────────────────────────────────────
STITCH_REGION = True
N_SAMPLE      = 100      # ignored when STITCH_REGION=True; set None for all patches
SELECTED_PATCHES = []
# ───────────────────────────────────────────────────────────────────────────────

PATCH_SIZE   = 256
STITCH_MAX_Y = 4352
STITCH_MAX_X = 4352

INPUT_FOLDER = Path("/kaggle/input/datasets/harish77718/darmstadt-dop20-presliced/darmstadt_dop20/images")
VIS_DIR  = Path("/kaggle/working/hessen_vis");  VIS_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = Path("/kaggle/working/hessen_preds"); PRED_DIR.mkdir(parents=True, exist_ok=True)

all_paths = sorted(INPUT_FOLDER.glob("*.png"))
if not all_paths:
    print(f"No images found at {INPUT_FOLDER}")
    for p in sorted(INPUT_FOLDER.parent.rglob("*"))[:20]: print(" ", p)
    raise SystemExit(1)

coord_map = {}
for p in all_paths:
    m = re.search(r'_y(\d+)_x(\d+)', p.stem)
    if m:
        coord_map[(int(m.group(1)), int(m.group(2)))] = p

if SELECTED_PATCHES:
    stems = {Path(s).stem for s in SELECTED_PATCHES}
    image_paths = [p for p in all_paths if p.stem in stems]
    vis_indices = set(range(len(image_paths)))
    print(f"Running {len(image_paths)} selected patch(es).")

elif STITCH_REGION and coord_map:
    y_vals = sorted(set(y for y, x in coord_map))
    x_vals = sorted(set(x for y, x in coord_map))
    y_step = min(y_vals[i+1]-y_vals[i] for i in range(len(y_vals)-1)) if len(y_vals) > 1 else PATCH_SIZE
    x_step = min(x_vals[i+1]-x_vals[i] for i in range(len(x_vals)-1)) if len(x_vals) > 1 else PATCH_SIZE
    print(f"Grid detected: y={y_vals[0]}–{y_vals[-1]} step={y_step}, x={x_vals[0]}–{x_vals[-1]} step={x_step}")
    skip_y = PATCH_SIZE // y_step
    skip_x = PATCH_SIZE // x_step
    stitch_y_vals = [y for i, y in enumerate(y_vals) if i % skip_y == 0 and y <= STITCH_MAX_Y]
    stitch_x_vals = [x for i, x in enumerate(x_vals) if i % skip_x == 0 and x <= STITCH_MAX_X]
    image_paths = [coord_map[(y, x)] for y in stitch_y_vals for x in stitch_x_vals if (y, x) in coord_map]
    vis_indices = set(range(len(image_paths)))
    print(f"Stitch grid: {len(stitch_y_vals)}×{len(stitch_x_vals)} = {len(image_paths)} patches")
    est_min = len(image_paths) * 5 / 60
    print(f"Estimated time on T4: {est_min:.0f}–{est_min*1.5:.0f} min")

elif N_SAMPLE is not None:
    step = max(1, len(all_paths) // N_SAMPLE)
    image_paths = all_paths[::step][:N_SAMPLE]
    vis_indices = set(range(len(image_paths)))
    print(f"Running {len(image_paths)} sampled patches.")

else:
    image_paths = all_paths
    step_vis = max(1, len(image_paths) // 20)
    vis_indices = set(range(0, len(image_paths), step_vis)[:20])
    print(f"Running ALL {len(image_paths)} patches — visualizing {len(vis_indices)} evenly-spaced.")

# 7-class urban vocabulary matching teammate's working config
CLASS_NAMES = [
    "impervious paved surface", "building", "dense tree canopy",
    "automotive vehicle", "low vegetation, grass", "railway tracks", "paved road"
]
COLOR_MAP = np.array([
    [128, 128, 128],   # 0 impervious — grey
    [  0,   0, 255],   # 1 building   — blue
    [  0, 200,   0],   # 2 tree canopy — green
    [255, 255,   0],   # 3 vehicle    — yellow
    [  0, 255, 255],   # 4 vegetation — cyan
    [255,   0, 255],   # 5 railway    — magenta
    [ 80,  80,  80],   # 6 road       — dark grey
], dtype=np.uint8)

# bg_idx=255, confidence_threshold=0.15, slide_crop=512 → single-view per 512px patch
# (no sliding window overhead; T4 bfloat16 autocast active inside _inference_single_view)
model = SegEarthOV3Segmentation(
    type="SegEarthOV3Segmentation",
    classname_path="./configs/cls_hessen.txt",
    prob_thd=0.1,
    bg_idx=255,
    confidence_threshold=0.15,
    slide_stride=448,
    slide_crop=512,
)

# Skip patches whose .npy already exists — safe to rerun after partial failure
todo = [p for p in image_paths if not (PRED_DIR / f"{p.stem}_pred.npy").exists()]
done_count = len(image_paths) - len(todo)
if done_count:
    print(f"Skipping {done_count} already-predicted patches. Running {len(todo)} remaining.")

for idx, img_path in enumerate(tqdm(todo, desc="Hessen inference")):
    t0 = time.time()
    img = Image.open(img_path).convert("RGB")
    ds = SegDataSample()
    ds.set_metainfo({"img_path": str(img_path), "ori_shape": img.size[::-1]})
    img_tensor = transforms.ToTensor()(img).unsqueeze(0).to("cuda")
    result = model.predict(img_tensor, data_samples=[ds])
    seg_pred = result[0].pred_sem_seg.data.cpu().numpy().squeeze(0)

    np.save(str(PRED_DIR / f"{img_path.stem}_pred.npy"), seg_pred.astype(np.uint8))

    global_idx = done_count + idx
    if global_idx in vis_indices:
        # bg_idx=255 → map unknown pixels to impervious (grey) before palette lookup
        safe_pred = np.where(seg_pred == 255, 0, seg_pred)
        seg_rgb = COLOR_MAP[np.clip(safe_pred, 0, len(COLOR_MAP) - 1)]
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(img); axes[0].axis("off"); axes[0].set_title("RGB (Hessen DOP20)")
        axes[1].imshow(img); axes[1].imshow(seg_rgb, alpha=0.5)
        axes[1].axis("off"); axes[1].set_title("Prediction — cls_hessen (7 classes)")
        fig.legend(
            handles=[Patch(facecolor=c/255.0, label=n) for n, c in zip(CLASS_NAMES, COLOR_MAP)],
            loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.04)
        )
        plt.tight_layout(rect=[0, 0.07, 1, 1])
        plt.savefig(VIS_DIR / f"{img_path.stem}.png", dpi=100, bbox_inches="tight")
        plt.close()

    elapsed = time.time() - t0
    remaining = (len(todo) - idx - 1) * elapsed
    print(f"  [{global_idx+1}/{len(image_paths)}] {img_path.name} → {elapsed:.1f}s  (ETA {remaining/60:.1f} min)")

print(f"\nInference done. {len(image_paths)} patches. Predictions in {PRED_DIR}")
print("Run the next cell (3b) to stitch — no re-inference needed if it fails.")
EOF

In [ ]:
## 3b — Stitch + dual coarse/fine output (rerun alone — no re-inference needed)
import re, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from PIL import Image

PATCH_SIZE   = 256
STITCH_MAX_Y = 4352
STITCH_MAX_X = 4352

INPUT_FOLDER = Path('/kaggle/input/datasets/harish77718/darmstadt-dop20-presliced/darmstadt_dop20/images')
PRED_DIR     = Path('/kaggle/working/hessen_preds')

# Fine 7-class palette
CLASS_NAMES = [
    'impervious paved surface', 'building', 'dense tree canopy',
    'automotive vehicle', 'low vegetation / grass', 'railway tracks', 'paved road'
]
COLOR_MAP = np.array([
    [128, 128, 128],
    [  0,   0, 255],
    [  0, 200,   0],
    [255, 255,   0],
    [  0, 255, 255],
    [255,   0, 255],
    [ 80,  80,  80],
], dtype=np.uint8)

# Coarse 4-class merge: impervious→0, building→1, tree→2, vehicle→3,
#                        low-veg→2, railway→0, road→0
FINE_TO_COARSE   = np.array([0, 1, 2, 3, 2, 0, 0], dtype=np.uint8)
COARSE_NAMES     = ['impervious & transport', 'built structure', 'vegetation', 'vehicle']
COARSE_COLOR_MAP = np.array([
    [128, 128, 128],
    [  0,   0, 255],
    [  0, 180,   0],
    [255, 255,   0],
], dtype=np.uint8)

pred_files = sorted(PRED_DIR.glob('*_pred.npy'))
if not pred_files:
    print('No .npy predictions found — run the inference cell first.')
else:
    coord_map = {}
    for pf in pred_files:
        m = __import__('re').search(r'_y(\d+)_x(\d+)_pred', pf.stem)
        if m:
            coord_map[(int(m.group(1)), int(m.group(2)))] = pf

    coords = [(y, x, pf) for (y, x), pf in coord_map.items()
              if y <= STITCH_MAX_Y and x <= STITCH_MAX_X]
    if not coords:
        print('No patches in stitch region.')
    else:
        h_out = max(y for y, x, _ in coords) + PATCH_SIZE
        w_out = max(x for y, x, _ in coords) + PATCH_SIZE
        rgb_canvas    = np.zeros((h_out, w_out, 3), dtype=np.uint8)
        fine_canvas   = np.zeros((h_out, w_out, 3), dtype=np.uint8)
        coarse_canvas = np.zeros((h_out, w_out, 3), dtype=np.uint8)
        pred_index    = np.full((h_out, w_out), 255, dtype=np.uint8)

        for y, x, pf in sorted(coords):
            stem = pf.stem.replace('_pred', '')
            img_path = next(INPUT_FOLDER.glob(f'{stem}.png'), None)
            if img_path is None:
                continue
            img_arr  = np.array(Image.open(img_path).convert('RGB'))[:PATCH_SIZE, :PATCH_SIZE]
            seg_pred = np.load(str(pf))[:PATCH_SIZE, :PATCH_SIZE]
            safe_pred   = np.where(seg_pred == 255, 0, seg_pred)
            fine_rgb    = COLOR_MAP[np.clip(safe_pred, 0, len(COLOR_MAP) - 1)]
            coarse_idx  = FINE_TO_COARSE[np.clip(safe_pred, 0, len(FINE_TO_COARSE) - 1)]
            coarse_rgb  = COARSE_COLOR_MAP[coarse_idx]
            rgb_canvas   [y:y+PATCH_SIZE, x:x+PATCH_SIZE] = img_arr
            fine_canvas  [y:y+PATCH_SIZE, x:x+PATCH_SIZE] = fine_rgb
            coarse_canvas[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = coarse_rgb
            pred_index   [y:y+PATCH_SIZE, x:x+PATCH_SIZE] = seg_pred

        np.save('/kaggle/working/hessen_pred_index.npy', pred_index)

        for canvas, names, cmap, suffix, title in [
            (fine_canvas,   CLASS_NAMES,   COLOR_MAP,        'fine',   '7-class fine'),
            (coarse_canvas, COARSE_NAMES,  COARSE_COLOR_MAP, 'coarse', '4-class coarse'),
        ]:
            fig, axes = plt.subplots(1, 2, figsize=(28, 14))
            axes[0].imshow(rgb_canvas); axes[0].axis('off')
            axes[0].set_title('Hessen DOP20 (stitched RGB)', fontsize=14)
            axes[1].imshow(canvas);     axes[1].axis('off')
            axes[1].set_title(f'SegEarth-OV-3 — {title} (stitched)', fontsize=14)
            fig.legend(
                handles=[Patch(facecolor=c/255.0, label=n) for n, c in zip(names, cmap)],
                loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.01), fontsize=11
            )
            plt.tight_layout(rect=[0, 0.04, 1, 1])
            plt.savefig(f'/kaggle/working/hessen_stitched_{suffix}.png', dpi=100, bbox_inches='tight')
            plt.show(); plt.close()
            Image.fromarray(canvas).save(f'/kaggle/working/hessen_stitched_pred_{suffix}.png')

        Image.fromarray(rgb_canvas).save('/kaggle/working/hessen_stitched_rgb.png')
        flat = pred_index[pred_index != 255]
        print(f'\nStitched {w_out}x{h_out}px from {len(coords)} patches.')
        print('Saved: hessen_stitched_fine.png  hessen_stitched_coarse.png  hessen_pred_index.npy')
        print('\nPer-class pixel coverage (fine):')
        for i, name in enumerate(CLASS_NAMES):
            pct = (flat == i).sum() / flat.size * 100
            print(f'  {name:<35} {pct:5.1f}%')
        print(f'  {"unknown (bg_idx=255)":<35} {(pred_index==255).sum()/pred_index.size*100:5.1f}%')


In [ ]:
## 3c — Single-source mosaic (optional deep-dive on one base image)
import re, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from PIL import Image

SOURCE_IMAGE = "dop20_32_474_5532_1_he"  # stem without _y_x suffix
PATCH_SIZE   = 256

INPUT_FOLDER = Path("/kaggle/input/datasets/harish77718/darmstadt-dop20-presliced/darmstadt_dop20/images")
PRED_DIR     = Path("/kaggle/working/hessen_preds")

CLASS_NAMES = [
    "impervious paved surface", "building", "dense tree canopy",
    "automotive vehicle", "low vegetation, grass", "railway tracks", "paved road"
]
COLOR_MAP = np.array([
    [128, 128, 128],
    [  0,   0, 255],
    [  0, 200,   0],
    [255, 255,   0],
    [  0, 255, 255],
    [255,   0, 255],
    [ 80,  80,  80],
], dtype=np.uint8)

pred_files = sorted(PRED_DIR.glob(f"{SOURCE_IMAGE}_y*_x*_pred.npy"))
if not pred_files:
    print("No predictions found — run inference cell first.")
else:
    coords = []
    for pf in pred_files:
        m = re.search(r"_y(\d+)_x(\d+)_pred", pf.name)
        if m:
            coords.append((int(m.group(1)), int(m.group(2)), pf))

    H = max(y for y, x, _ in coords) + PATCH_SIZE
    W = max(x for y, x, _ in coords) + PATCH_SIZE
    rgb_canvas  = np.zeros((H, W, 3), dtype=np.uint8)
    pred_canvas = np.zeros((H, W, 3), dtype=np.uint8)

    for y, x, pf in coords:
        stem = pf.stem.replace("_pred", "")
        img_path = INPUT_FOLDER / f"{stem}.png"
        img      = np.array(Image.open(img_path).convert("RGB"))[:PATCH_SIZE, :PATCH_SIZE]
        seg_pred = np.load(str(pf))[:PATCH_SIZE, :PATCH_SIZE]
        safe_pred = np.where(seg_pred == 255, 0, seg_pred)
        pred_rgb  = COLOR_MAP[np.clip(safe_pred, 0, len(COLOR_MAP)-1)]
        rgb_canvas [y:y+PATCH_SIZE, x:x+PATCH_SIZE] = img
        pred_canvas[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = pred_rgb

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(rgb_canvas);  axes[0].set_title(f"RGB — {SOURCE_IMAGE}", fontsize=13); axes[0].axis("off")
    axes[1].imshow(rgb_canvas);  axes[1].imshow(pred_canvas, alpha=0.55)
    axes[1].set_title("Prediction overlay — cls_hessen (7 classes)", fontsize=13); axes[1].axis("off")
    fig.legend(
        handles=[Patch(facecolor=c/255.0, label=n) for n, c in zip(CLASS_NAMES, COLOR_MAP)],
        loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01), fontsize=11
    )
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    out = Path(f"/kaggle/working/{SOURCE_IMAGE}_mosaic.png")
    plt.savefig(out, dpi=80, bbox_inches="tight")
    plt.show()
    print(f"Mosaic saved: {out}  ({W}×{H}px from {len(coords)} patches)")

## 4 — OSM pseudo-GT F1 (optional)

Requires OSM masks to be available at `/kaggle/working/osm_masks.npz`.
Generate them first using `segearth_utils.osm_eval.build_osm_masks()`
or download the pre-built file from HF if available.

Always report as **"OSM pseudo-GT F1"** — not ground-truth F1.

In [ ]:
%%bash
# Install OSM tools into the segearth conda env
source /tmp/miniconda/bin/activate segearth
pip install osmnx rasterio affine scipy -q
python -c "import osmnx; import rasterio; import affine; print('OSM libs OK')"


In [ ]:
## 4a — Generate OSM pseudo-GT masks for Darmstadt
#
# Rasterizes OSM polygons to 256x256 masks aligned to each DOP20 patch.
# Saves /kaggle/working/osm_masks.npz  (stem -> int16 mask, 255=ignore)
#
# Class mapping (NB03 cls_hessen.txt line order = class index):
#   0 impervious  -- left as 255 (OSM impervious coverage is incomplete)
#   1 building    -- OSM building=*
#   2 tree canopy -- OSM natural=wood, landuse=forest
#   3 vehicle     -- left as 255 (not in OSM static data)
#   4 vegetation  -- OSM landuse=grass/meadow, natural=grassland/scrub
#   5 railway     -- OSM railway=* buffered 3m
#   6 road        -- OSM highway=* buffered by road type
#
# NOTE: MOSAIC_NORTH is estimated from tile name 474_5532 (SW corner = 5532000N).
# Adjust if alignment looks off by checking a known landmark in the stitched output.

import numpy as np, re, warnings, pandas as pd
from pathlib import Path
import osmnx as ox
import rasterio.features as rio_features
from affine import Affine
from scipy.ndimage import binary_erosion

warnings.filterwarnings('ignore')

MOSAIC_WEST  = 474000.0   # easting of mosaic pixel x=0
MOSAIC_NORTH = 5533000.0  # northing of mosaic pixel y=0 (top edge)
PIXEL_SIZE   = 0.20       # 20cm GSD
PATCH_SIZE   = 256

PRED_DIR       = Path('/kaggle/working/hessen_preds')
OSM_MASKS_PATH = Path('/kaggle/working/osm_masks.npz')

pred_files = sorted(PRED_DIR.glob('*_pred.npy'))
if not pred_files:
    print('No predictions found — run inference cell first.')
    raise SystemExit(0)

stems, coords = [], {}
for pf in pred_files:
    m = re.search(r'_y(\d+)_x(\d+)_pred', pf.stem)
    if m:
        stem = pf.stem.replace('_pred', '')
        stems.append(stem)
        coords[stem] = (int(m.group(1)), int(m.group(2)))
print(f'Patches to rasterize: {len(stems)}')

print('Downloading OSM features for Darmstadt...')
OSM_TAGS = {
    'building': True,
    'highway': True,
    'railway': True,
    'natural': ['wood', 'grassland', 'scrub', 'heath'],
    'landuse': ['forest', 'grass', 'meadow', 'farmland'],
}
gdf = ox.features_from_place('Darmstadt, Germany', tags=OSM_TAGS).to_crs(epsg=25832)
print(f'Downloaded {len(gdf)} OSM features.')

def get_class_geoms(gdf):
    g = {}
    g[1] = list(gdf[gdf.get('building', pd.Series(dtype=str)).notna()].geometry.dropna())
    g[2] = list(gdf[
        (gdf.get('natural', pd.Series(dtype=str)) == 'wood') |
        (gdf.get('landuse', pd.Series(dtype=str)) == 'forest')
    ].geometry.dropna())
    g[4] = list(gdf[
        gdf.get('natural', pd.Series(dtype=str)).isin(['grassland', 'scrub', 'heath']) |
        gdf.get('landuse', pd.Series(dtype=str)).isin(['grass', 'meadow', 'farmland'])
    ].geometry.dropna())
    rwy = gdf[gdf.get('railway', pd.Series(dtype=str)).notna()]
    g[5] = [h.buffer(3.0) if h.geom_type in ('LineString','MultiLineString') else h
            for h in rwy.geometry.dropna()]
    HW_BUF = {'motorway':8,'trunk':7,'primary':6,'secondary':5,
               'tertiary':4,'residential':3,'service':2,'track':2}
    hw = gdf[gdf.get('highway', pd.Series(dtype=str)).notna()]
    g[6] = [row.geometry.buffer(HW_BUF.get(str(row.get('highway','service')),2))
            if row.geometry and row.geometry.geom_type in ('LineString','MultiLineString')
            else row.geometry
            for _, row in hw.iterrows() if row.geometry is not None]
    return g

class_geoms = get_class_geoms(gdf)
for cls, gs in class_geoms.items():
    print(f'  Class {cls}: {len(gs)} geometries')

def erode_mask(mask, k=3):
    eroded = mask.copy()
    n = max(1, (k - 1) // 2)
    for cls in [c for c in np.unique(mask) if c != 255]:
        cm = (mask == cls)
        eroded[~binary_erosion(cm, iterations=n) & cm] = 255
    return eroded

print('\nRasterizing patches...')
burn_order = [4, 2, 5, 6, 1]  # buildings last = highest priority
osm_masks = {}
for i, stem in enumerate(stems):
    y_px, x_px = coords[stem]
    tf = Affine(PIXEL_SIZE, 0, MOSAIC_WEST + x_px * PIXEL_SIZE,
                0, -PIXEL_SIZE, MOSAIC_NORTH - y_px * PIXEL_SIZE)
    mask = np.full((PATCH_SIZE, PATCH_SIZE), 255, dtype=np.int16)
    for cls in burn_order:
        if class_geoms.get(cls):
            rio_features.rasterize(
                ((geom, cls) for geom in class_geoms[cls]),
                out=mask, transform=tf, dtype=np.int16
            )
    osm_masks[stem] = erode_mask(mask)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(stems)} done')

np.savez_compressed(str(OSM_MASKS_PATH), **osm_masks)
total_lbl = sum((v != 255).sum() for v in osm_masks.values())
total_px  = sum(v.size for v in osm_masks.values())
print(f'\nSaved {len(osm_masks)} OSM masks -> {OSM_MASKS_PATH}')
print(f'Coverage: {total_lbl/total_px*100:.1f}% of pixels labeled (rest = 255/ignore)')
print('Run cell 4b (osm-eval) to compute pseudo-GT F1.')


In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'EOF'
import numpy as np
from pathlib import Path

OSM_MASKS_PATH = Path("/kaggle/working/osm_masks.npz")
PREDS_DIR      = Path("/kaggle/working/hessen_preds")

if not OSM_MASKS_PATH.exists():
    print("OSM masks not found — skipping F1 eval.")
    raise SystemExit(0)

osm_data = np.load(str(OSM_MASKS_PATH))
osm_masks = {k: osm_data[k].astype(np.int64) for k in osm_data.files}

pred_files = sorted(PREDS_DIR.glob("*_pred.npy"))
pred_masks = {p.stem.replace("_pred", ""): np.load(str(p)).astype(np.int64)
              for p in pred_files}

CLASS_NAMES = ["agricultural field", "forest", "building", "road", "water body", "background"]

stems = sorted(set(pred_masks) & set(osm_masks))
print(f"Evaluating {len(stems)} patches...")

n_cls = 6
ignore_id = 255
tp = np.zeros(n_cls); fp = np.zeros(n_cls); fn = np.zeros(n_cls)
for stem in stems:
    pred = pred_masks[stem]
    gt   = osm_masks[stem]
    valid = gt != ignore_id
    for cls in range(n_cls):
        pc = (pred == cls) & valid
        lc = (gt   == cls) & valid
        tp[cls] += (pc & lc).sum()
        fp[cls] += (pc & ~lc).sum()
        fn[cls] += (~pc & lc).sum()

prec = tp / (tp + fp + 1e-6)
rec  = tp / (tp + fn + 1e-6)
f1   = 2 * prec * rec / (prec + rec + 1e-6)

print("\n=== OSM pseudo-GT F1 ===")
print(f"{'Class':<22} {'F1':>6}")
print("-" * 30)
for cls, (name, score) in enumerate(zip(CLASS_NAMES, f1)):
    print(f"{name:<22} {score:>6.4f}")
print("-" * 30)
print(f"{'MEAN':<22} {f1.mean():>6.4f}")
print(f"\nPatches evaluated: {len(stems)}")
EOF